### Exercise 3 - Preparing Real-World Data for a Neural Network

A - Get to know the data

1. Download the Spaceship Titanic dataset

In [ ]:
from pathlib import Path
import kagglehub

data_dir = Path("./data")
data_dir.mkdir(parents=True, exist_ok=True)

dataset_path = kagglehub.competition_download(
    "spaceship-titanic",
    output_dir=str(data_dir),
)

print("Dataset downloaded to:", dataset_path)
print("Files:", [file.name for file in data_dir.iterdir()])

2. Describe the goal of the dataset: what does the ``Transported`` column represent? What is the class balance between the two labels?

    **A:** The goal of the Spaceship Titanic dataset is to predict whether a passenger was transported to an alternate dimension after the spaceship collided with a spacetime anomaly. The `Transported` column is the binary target variable:

    - `True`: the passenger was transported.
    - `False`: the passenger was not transported.

    The training dataset contains 8,693 passengers. Its class distribution is:

    | Transported | Count | Percentage |
    |---|---:|---:|
    | `True` | 4,378 | 50.36% |
    | `False` | 4,315 | 49.64% |

    Therefore, the dataset is very well balanced, with only a small difference of 63 passengers between the two classes. No major class-imbalance correction is necessary.

In [4]:
import pandas as pd

train_df = pd.read_csv(data_dir / "train.csv")

class_counts = train_df["Transported"].value_counts()
class_percentages = (
    train_df["Transported"]
    .value_counts(normalize=True)
    .mul(100)
)

class_balance = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages,
})

print(class_balance)

             Count  Percentage
Transported                   
True          4378   50.362361
False         4315   49.637639


3. List the features, separating numerical from categorical.

    **A:** The dataset contains the following input features:

    Numerical features

    - `Age`: passenger's age.
    - `RoomService`: amount spent on room service.
    - `FoodCourt`: amount spent at the food court.
    - `ShoppingMall`: amount spent at the shopping mall.
    - `Spa`: amount spent at the spa.
    - `VRDeck`: amount spent on the virtual-reality deck.

    Categorical features

    - `HomePlanet`: passenger's planet of origin.
    - `CryoSleep`: whether the passenger was placed in suspended animation.
    - `Cabin`: cabin number in the format `deck/number/side`.
    - `Destination`: passenger's destination planet.
    - `VIP`: whether the passenger paid for VIP service.

    Identifier and text columns

    - `PassengerId`: unique passenger identifier in the format `group_number/passenger_number`.
    - `Name`: passenger's name.

    `PassengerId` and `Name` are not ordinary categorical features because most of their values are unique. However, useful information can be extracted from them, such as the passenger's group from `PassengerId` or family information from `Name`.

    The `Transported` column is the target and is therefore not included among the input features.

4. Build a table of missing values per column, in absolute count and in percentage.

In [5]:
missing_values = (
    pd.DataFrame({
        "Missing values": train_df.isna().sum(),
        "Missing percentage": train_df.isna().mean() * 100,
    })
    .sort_values("Missing values", ascending=False)
    .rename_axis("Column")
    .reset_index()
)

missing_values["Missing percentage"] = (
    missing_values["Missing percentage"].round(2)
)

display(missing_values)

,Column,Missing values,Missing percentage
0,CryoSleep,217,2.50
1,ShoppingMall,208,2.39
2,VIP,203,2.34
3,HomePlanet,201,2.31
4,Name,200,2.30
5,Cabin,199,2.29
6,VRDeck,188,2.16
7,Spa,183,2.11
8,FoodCourt,183,2.11
9,Destination,182,2.09


5. For the spending columns, report mean, median, and maximum. Compare mean and median: what does that difference tell you about the spread and the skewness of those distributions?

In [7]:
spending_columns = [
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
]

spending_summary = (
    train_df[spending_columns]
    .agg(["mean", "median", "max"])
    .T
    .rename(columns={
        "mean": "Mean",
        "median": "Median",
        "max": "Maximum",
    })
    .round(2)
)

display(spending_summary)

,Mean,Median,Maximum
RoomService,224.69,0.0,14327.0
FoodCourt,458.08,0.0,29813.0
ShoppingMall,173.73,0.0,23492.0
Spa,311.14,0.0,22408.0
VRDeck,304.85,0.0,24133.0


All five spending features have a median of zero, indicating that at least half of the passengers did not spend money in each respective category. However, their means are considerably greater than zero, and their maximum values are extremely high.

The large difference between the mean and median indicates that the distributions are strongly right-skewed: most passengers spent little or nothing, while a relatively small number of passengers spent very large amounts. These large values pull the mean upward.

The wide range between zero and the maximum values also indicates substantial spread and the presence of extreme values. `FoodCourt` has the highest mean and maximum spending among the five features.

The mean–median difference primarily indicates skewness rather than measuring spread directly. Measures such as the standard deviation or interquartile range would be needed to quantify the spread more precisely.

B - Split before you transform

In [8]:
from sklearn.model_selection import train_test_split

X = train_df.drop(columns=["Transported"])
y = train_df["Transported"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).sort_index())

X_train: (6954, 13)
X_test: (1739, 13)
y_train: (6954,)
y_test: (1739,)

Training target distribution:
Transported
0    0.496405
1    0.503595
Name: proportion, dtype: float64

Test target distribution:
Transported
0    0.496262
1    0.503738
Name: proportion, dtype: float64


Explain, in two or three sentences, why this split comes before imputation and scaling.

**A:** The split must be performed before imputation and scaling so that these preprocessing steps are fitted using only the training data. Otherwise, information from the test set could influence the imputed values and scaling parameters, causing data leakage and producing an overly optimistic evaluation of the model.

C - Preprocess 